# 21.8 — SAT solvers

SAT solving scales truth tables by searching Boolean assignments with propagation, pruning, and backtracking.

SAT solvers industrialize the truth-table idea from propositional logic. SMT and ASP build on SAT by adding theories or stable-model semantics above the Boolean core.

Save a copy to Drive to edit.

In [ ]:

import itertools
import math
import random

import matplotlib.pyplot as plt

random.seed(218)


def lit_var(literal):
    return literal[1:] if literal.startswith("~") else literal


def lit_neg(literal):
    return literal[1:] if literal.startswith("~") else "~" + literal


def eval_lit(literal, assignment):
    variable = lit_var(literal)
    value = assignment.get(variable)
    if value is None:
        return None
    if literal.startswith("~"):
        return not value
    return value


def simplify_cnf(cnf, literal):
    opposite = lit_neg(literal)
    simplified = []
    for clause in cnf:
        if literal in clause:
            continue
        new_clause = [item for item in clause if item != opposite]
        simplified.append(new_clause)
    return simplified


def brute_force_count(cnf, variables):
    satisfying = []
    for values in itertools.product([False, True], repeat=len(variables)):
        assignment = dict(zip(variables, values))
        ok = True
        for clause in cnf:
            clause_ok = any(eval_lit(literal, assignment) for literal in clause)
            if not clause_ok:
                ok = False
        if ok:
            satisfying.append(assignment)
    return satisfying


def build_sat_ladder():
    rungs = []
    rungs.append({
        "name": "D1 tiny 3-variable CNF",
        "variables": ["A", "B", "C"],
        "cnf": [["A", "B"], ["~A", "C"]],
        "expected_sat": True,
    })
    rungs.append({
        "name": "D2 implication chain",
        "variables": ["A", "B", "C", "D", "E"],
        "cnf": [["A"], ["~A", "B"], ["~B", "C"], ["~C", "D"], ["~D", "E"]],
        "expected_sat": True,
    })
    rungs.append({
        "name": "D3 conflicting unit clauses",
        "variables": ["A", "B", "C", "D", "E", "F"],
        "cnf": [["A"], ["~A", "B"], ["~B", "C"], ["~C"], ["D", "E"], ["~D", "F"]],
        "expected_sat": False,
    })
    rungs.append({
        "name": "D4 package configuration",
        "variables": ["core", "ui", "gpu", "cpu", "net", "db", "cache", "lite", "pro", "test"],
        "cnf": [["core"], ["~ui", "core"], ["~gpu", "core"], ["~gpu", "~lite"], ["~pro", "db"], ["~db", "net"], ["~cache", "db"], ["ui", "lite", "pro"], ["~cpu", "~gpu"], ["cpu", "gpu"], ["~test", "cache"]],
        "expected_sat": True,
    })
    variables = [f"X{i}" for i in range(1, 21)]
    cnf = [["X1"]]
    for index in range(1, 20):
        cnf.append([f"~X{index}", f"X{index + 1}"])
    cnf.extend([
        ["S1", "S2"],
        ["~S1", "~S2"],
        ["~X20", "Y1"],
        ["~Y1", "Y2"],
        ["~Y2", "Y3"],
        ["~Y3", "Z"],
    ])
    variables = variables + ["S1", "S2", "Y1", "Y2", "Y3", "Z"]
    rungs.append({
        "name": "D5 scheduling CNF with deep propagation",
        "variables": variables,
        "cnf": cnf,
        "expected_sat": True,
    })
    return rungs


## The concept, built once (D1)

A CNF formula is a conjunction of clauses, each clause is a disjunction of literals. The lesson formula is $(A\lor B)\land(\neg A\lor C)$ over three variables, so brute force has $2^3=8$ rows and exactly $4$ satisfying rows.

First build the truth-table check that DPLL is meant to avoid.

In [ ]:

lesson_cnf = [["A", "B"], ["~A", "C"]]
lesson_variables = ["A", "B", "C"]
lesson_solutions = brute_force_count(lesson_cnf, lesson_variables)
lesson_rows = 2 ** len(lesson_variables)

assert lesson_rows == 8
assert len(lesson_solutions) == 4

print("truth-table rows", lesson_rows)
print("satisfying rows", len(lesson_solutions))


Now implement real DPLL: repeatedly apply unit propagation, use pure literals, and branch only when forced assignments stop. The unit clause $(A)$ forces $A=1$; after substitution, $(\neg A\lor C)$ becomes unit $(C)$.

In [ ]:

def dpll(cnf, assignment=None, stats=None, trace=None):
    if assignment is None:
        assignment = {}
    if stats is None:
        stats = {"nodes": 0, "forced": 0, "branches": 0}
    if trace is None:
        trace = []
    stats["nodes"] += 1
    trace.append((len(trace), "enter", dict(assignment), len(cnf)))
    while True:
        if not cnf:
            return True, assignment, stats, trace
        if any(len(clause) == 0 for clause in cnf):
            trace.append((len(trace), "conflict", dict(assignment), len(cnf)))
            return False, assignment, stats, trace
        unit_literal = None
        for clause in cnf:
            if len(clause) == 1:
                unit_literal = clause[0]
                break
        if unit_literal is not None:
            variable = lit_var(unit_literal)
            assignment[variable] = not unit_literal.startswith("~")
            stats["forced"] += 1
            trace.append((len(trace), "unit " + unit_literal, dict(assignment), len(cnf)))
            cnf = simplify_cnf(cnf, unit_literal)
            continue
        counts = {}
        for clause in cnf:
            for literal in clause:
                counts[literal] = counts.get(literal, 0) + 1
        pure_literal = None
        for literal in sorted(counts):
            if lit_neg(literal) not in counts:
                pure_literal = literal
                break
        if pure_literal is not None:
            variable = lit_var(pure_literal)
            assignment[variable] = not pure_literal.startswith("~")
            stats["forced"] += 1
            trace.append((len(trace), "pure " + pure_literal, dict(assignment), len(cnf)))
            cnf = simplify_cnf(cnf, pure_literal)
            continue
        break
    branch_literal = sorted(cnf, key=len)[0][0]
    variable = lit_var(branch_literal)
    for value in [True, False]:
        stats["branches"] += 1
        chosen = variable if value else "~" + variable
        trace.append((len(trace), "branch " + chosen, dict(assignment), len(cnf)))
        new_assignment = dict(assignment)
        new_assignment[variable] = value
        sat, model, stats, trace = dpll(simplify_cnf(cnf, chosen), new_assignment, stats, trace)
        if sat:
            return True, model, stats, trace
    return False, assignment, stats, trace

sat, model, stats, trace = dpll(lesson_cnf)

assert sat is True
assert stats["nodes"] >= 1

print("DPLL model", model)
print("nodes", stats["nodes"])
print("forced", stats["forced"])


## The dataset ladder

D1–D5 are inline F16 algorithmic instances with rising variables, clauses, propagation depth, and conflict structure.

In [ ]:

sat_rungs = build_sat_ladder()

for rung in sat_rungs:
    print(rung["name"])
    print("  variables", len(rung["variables"]))
    print("  clauses", len(rung["cnf"]))
    print("  sample", rung["cnf"][:3])


## Run the same method across D1–D5

In [ ]:

sat_results = []

for rung in sat_rungs:
    sat, model, stats, trace = dpll([list(clause) for clause in rung["cnf"]])
    assert sat == rung["expected_sat"]
    sat_results.append({
        "name": rung["name"],
        "variables": len(rung["variables"]),
        "clauses": len(rung["cnf"]),
        "sat": sat,
        "nodes": stats["nodes"],
        "forced": stats["forced"],
        "trace": trace,
    })

print("rung | sat | vars | clauses | nodes | forced")
for result in sat_results:
    print(result["name"], result["sat"], result["variables"], result["clauses"], result["nodes"], result["forced"])


## Results visualization

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(18, 7))

for index, result in enumerate(sat_results):
    ax = axes[0][index]
    events = result["trace"][:8]
    ax.set_title("D" + str(index + 1))
    ax.axis("off")
    lines = []
    for event in events:
        lines.append(str(event[0]) + " " + event[1] + " clauses=" + str(event[3]))
    ax.text(0.02, 0.95, "\n".join(lines), va="top", family="monospace", fontsize=8)

x_values = list(range(1, 6))
nodes = [result["nodes"] for result in sat_results]
forced = [result["forced"] for result in sat_results]
axes[1][0].plot(x_values, nodes, marker="o", label="nodes")
axes[1][0].plot(x_values, forced, marker="s", label="forced")
axes[1][0].set_xticks(x_values)
axes[1][0].set_xlabel("rung")
axes[1][0].set_ylabel("count")
axes[1][0].set_title("steps vs size")
axes[1][0].legend()
for ax in axes[1][1:]:
    ax.axis("off")
plt.tight_layout()


## Pitfall on the hardest rung

Pitfall: choosing a representation that hides the scheduling constraint. A loose variable list cannot propagate exclusivity; explicit CNF clauses can.

In [ ]:

d5 = sat_rungs[-1]
hidden_requirements = ["slot must choose exactly one of S1 or S2"]
wrong_clauses = [clause for clause in d5["cnf"] if clause not in [["S1", "S2"], ["~S1", "~S2"]]]
wrong_sat, wrong_model, wrong_stats, wrong_trace = dpll([list(clause) for clause in wrong_clauses])
fixed_sat, fixed_model, fixed_stats, fixed_trace = dpll([list(clause) for clause in d5["cnf"]])

print("hidden requirements", hidden_requirements)
print("wrong clauses sat", wrong_sat, "model has S1", wrong_model.get("S1"), "S2", wrong_model.get("S2"))
print("fixed CNF sat", fixed_sat, "model has S1", fixed_model.get("S1"), "S2", fixed_model.get("S2"))
print("propagation steps after fix", fixed_stats["forced"])


## Evaluate it + Practice

- Metric: satisfiability, nodes visited, forced assignments.
- No-skill baseline: scan all Boolean rows.
- Cheap sanity check: D1 has 8 rows and 4 satisfying rows.
- Ablation: turn off unit propagation and nodes rise.
- Failure signals: a model violates a clause or the trace hides a forced literal.

Practice 1: Change the D2 instance by one constraint and predict the metric before running.

Practice 2: Add one contradictory or noisy condition to D4 and explain how the trace changes.

Practice 3: On D5, record the first step where pruning or explanation prevents a wrong answer.